# TRANSFER MATRIX METHOD

given a difference equation of the form $$ T_n \Psi_{n+1} = H_n \Psi{n} - B_n \Psi_{n-1} $$

where $\Psi_n$ is a M-dimensional column vector. Then we can write the above equation in a 2M x 2M matrix format as 
$$ \begin{pmatrix} \Psi_{n+1} \\ \Psi_{n} \end{pmatrix} = \begin{pmatrix} A_n^{-1} H_n & -A_n{-1} B_n \\ 1 & 0 \end{pmatrix} \begin{pmatrix} \Psi_n \\ \Psi_{n-1} \end{pmatrix}

## d-Density Wave TMM


In [3]:
import numpy as np
import matplotlib.pyplot as plt

### Creating Transfer Matrix 

In [ ]:
def TM_ddw(n, M, W0, E, phi, t, eps_matrix):

    tnn, tnnn = t

    An = np.zeros((M, M), dtype=complex)
    Tn = np.zeros((M, M), dtype=complex)
    Bn = np.zeros((M, M), dtype=complex)

    for m in range(M):
        m_p = (m + 1) % M
        m_m = (m - 1) % M
        stagger = (-1)**(n + m)

        # ---- A_n (intra-slice) ----
        An[m, m] = eps_matrix[n, m] - E

        hop = (-tnn + 0.25j * W0 * stagger)
        An[m, m_p] = hop * np.exp(-1j * n * phi)
        An[m_p, m] = np.conj(An[m, m_p])

        # ---- B_n (backward hopping) ----
        Bn[m, m] = tnn + 0.25j * W0 * stagger
        Bn[m, m_p] = -tnnn * np.exp(1j * (-n + 0.5) * phi)
        Bn[m, m_m] = -tnnn * np.exp(1j * (n - 0.5) * phi)

        # ---- T_n (forward hopping) ----
        Tn[m, m] = tnn + 0.25j * W0 * stagger
        Tn[m, m_p] = -tnnn * np.exp(1j * (-n - 0.5) * phi)
        Tn[m, m_m] = -tnnn * np.exp(1j * (n + 0.5) * phi)

    top_left = np.linalg.solve(Tn, An)
    top_right = -np.linalg.solve(Tn, Bn)

    T_full = np.block([
        [top_left, top_right],
        [np.eye(M), np.zeros((M, M))]
    ])

    return T_full

### Computing lyapunov exponents

Calculated using QR decomposition of the Transfer matrix, $T_n$

the product of the i-th diagonal entries of the $R_n$ matrices gives singular values of the $\mathcal{T}$, whose logarithm divided by N gives the i-th lyapunov exponent.

In [3]:
def lyapunov_exponents(N, M, TM_func, params, warmup_frac=0.1):

    dim = 2 * M
    Q = np.eye(dim, dtype=complex)

    lyap_sum = np.zeros(dim)
    count = 0

    eps = 1e-16  # numerical safety

    for n in range(N):
        Tn = TM_func(n, M, **params)

        # propagate basis
        Z = Tn @ Q

        # QR decomposition
        Q, R = np.linalg.qr(Z)

        # skip transient region
        if n > int(warmup_frac * N):
            lyap_sum += np.log(np.abs(np.diag(R)) + eps)
            count += 1

    lyap = lyap_sum / count

    return np.sort(lyap)


### Calculating Conductance

In [ ]:
def conductance(lyap):

    dim = len(lyap)
    M = dim // 2

    # take positive Lyapunov exponents i.e the first M
    gammas = lyap[M:]

    sigma = np.sum(1 / np.cosh(gammas)**2)

    return sigma

### Example Case

In [ ]:
if __name__ == "__main__":

    M = 32 #Width of slice
    N = 10000 #Number of slices

    W0 = 0.1 #DDW gap
    E = 0.0 #Chemical potential
    phi = 0.01 #Peierls phase

    tnn = 1.0 #nn hopping
    tnnn = 0.3 #nnn hopping

    # disorder
    eps_matrix = np.random.normal(0, 0.5, size=(N, M))

    params = {
        "W0": W0,
        "E": E,
        "phi": phi,
        "t": (tnn, tnnn),
        "eps_matrix": eps_matrix
    }

    lyap = lyapunov_exponents(N, M, TM_ddw, params)
    sigma = conductance(lyap)

    print("Lyapunov exponents:\n", lyap)
    print("Conductance:", sigma)

Lyapunov exponents:
 [-1.50851075e+00 -1.49512480e+00 -1.46182873e+00 -1.39269980e+00
 -1.28546657e+00 -1.15055087e+00 -9.82288186e-01 -7.57003436e-01
 -4.13761084e-01 -1.04113433e-01 -7.83167276e-02 -6.80949106e-02
 -5.96004694e-02 -5.40374053e-02 -4.93277640e-02 -4.50310888e-02
 -4.07119260e-02 -3.77479977e-02 -3.39832645e-02 -3.11665360e-02
 -2.79515625e-02 -2.56404511e-02 -2.25121852e-02 -2.05943574e-02
 -1.76762366e-02 -1.54517585e-02 -1.28342074e-02 -1.00610151e-02
 -8.03707517e-03 -5.94498253e-03 -3.63993779e-03 -1.43740515e-03
  1.50165904e-03  3.57561246e-03  5.97470414e-03  8.00472651e-03
  1.00773454e-02  1.28484272e-02  1.54477029e-02  1.76681068e-02
  2.05644382e-02  2.25234849e-02  2.56605467e-02  2.79394244e-02
  3.11752736e-02  3.39736436e-02  3.77031052e-02  4.07580612e-02
  4.50337745e-02  4.93246667e-02  5.40393411e-02  5.95951973e-02
  6.80878040e-02  7.83199927e-02  1.04109434e-01  4.13769460e-01
  7.56984499e-01  9.82305221e-01  1.15054304e+00  1.28547469e+00
  1.